In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ahmed101sahil/mcq-wikipedia-corpus/my_scraped_corpus/Speed_of_sound.txt
/kaggle/input/datasets/ahmed101sahil/mcq-wikipedia-corpus/my_scraped_corpus/Lift__force_.txt
/kaggle/input/datasets/ahmed101sahil/mcq-wikipedia-corpus/my_scraped_corpus/Evolutionary_history_of_plants.txt
/kaggle/input/datasets/ahmed101sahil/mcq-wikipedia-corpus/my_scraped_corpus/James_Webb_Space_Telescope.txt
/kaggle/input/datasets/ahmed101sahil/mcq-wikipedia-corpus/my_scraped_corpus/Spin_quantum_number.txt
/kaggle/input/datasets/ahmed101sahil/mcq-wikipedia-corpus/my_scraped_corpus/Medical_ultrasound.txt
/kaggle/input/datasets/ahmed101sahil/mcq-wikipedia-corpus/my_scraped_corpus/Penrose_process.txt
/kaggle/input/datasets/ahmed101sahil/mcq-wikipedia-corpus/my_scraped_corpus/Aviation_biofuel.txt
/kaggle/input/datasets/ahmed101sahil/mcq-wikipedia-corpus/my_scraped_corpus/Supermassive_black_hole.txt
/kaggle/input/datasets/ahmed101sahil/mcq-wikipedia-corpus/my_scraped_corpus/Total_internal_reflect

In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
WB_KEY = user_secrets.get_secret("wandb-key")


In [3]:
import wandb 
wandb.login(key=WB_KEY)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [5]:
import pandas as pd
import numpy as np
import re
import string

def load_data(train_path='/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv', 
              test_path='/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv'):
    
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    
    print(f"Train shape: {train_df.shape}")
    print(f"Test shape: {test_df.shape}")
    
    return train_df, test_df

train_df, test_df = load_data()


def clean_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text).lower()

    text = text.translate(str.maketrans('', '', string.punctuation))

    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

Train shape: (2000, 8)
Test shape: (500, 7)


In [6]:
text_columns = ['prompt', 'A', 'B', 'C', 'D', 'E']
for col in text_columns:
    train_df[f'clean_{col}'] = train_df[col].apply(clean_text)
    test_df[f'clean_{col}'] = test_df[col].apply(clean_text)

print("Text cleaning complete. New columns generated.")

Text cleaning complete. New columns generated.


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(max_features=5000, stop_words='english')

all_train_text = train_df[[f'clean_{col}' for col in text_columns]].astype(str).agg(' '.join, axis=1)
tfidf.fit(all_train_text)

print(f"TF-IDF Vocabulary size: {len(tfidf.vocabulary_)}")

TF-IDF Vocabulary size: 2865


In [9]:
# View as a dictionary
train_df[['prompt', 'A', 'B', 'C', 'D', 'E']].iloc[0].to_dict()

{'prompt': "Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.",
 'A': "Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time.",
 'B': 'Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.',
 'C': 'Martin Heidegger does not believe in the existence of time or that it has any effect on human consciousness. The relationship to the past and the future is insignificant, and human existence is solely based on the present.',
 'D': 'Martin Heidegger be

In [10]:
def get_top_3_tfidf(row, vectorizer):
    prompt_text = row['clean_prompt']
    options_text = [row['clean_A'], row['clean_B'], row['clean_C'], row['clean_D'], row['clean_E']]
    labels = ['A', 'B', 'C', 'D', 'E']
    
    prompt_vec = vectorizer.transform([prompt_text])
    options_vec = vectorizer.transform(options_text)
    similarities = cosine_similarity(prompt_vec, options_vec).flatten()
    top_3_idx = np.argsort(similarities)[::-1][:3]
    top_3_labels = [labels[i] for i in top_3_idx]
    return " ".join(top_3_labels)
train_df['tfidf_prediction'] = train_df.apply(lambda row: get_top_3_tfidf(row, tfidf), axis=1)

print("Predictions generated. Example output:")
print(train_df[['id', 'tfidf_prediction']].head(3))

Predictions generated. Example output:
   id tfidf_prediction
0   1            C D B
1   2            C A B
2   3            E D C


In [11]:
def calculate_map_at_3(true_labels, predicted_labels_list):
    scores = []
    
    for true_label, preds in zip(true_labels, predicted_labels_list):
        pred_list = preds.split() 
        
        if true_label in pred_list:
            # Find the rank (1-indexed)
            rank = pred_list.index(true_label) + 1
            scores.append(1.0 / rank)
        else:
            scores.append(0.0)
            
    return np.mean(scores)

baseline_map3 = calculate_map_at_3(train_df['answer'], train_df['tfidf_prediction'])
print(f"Baseline TF-IDF MAP@3 Score: {baseline_map3:.4f}")

Baseline TF-IDF MAP@3 Score: 0.2387


In [12]:
import numpy as np
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity

train_df['tokens_prompt'] = train_df['clean_prompt'].apply(lambda x: x.split())
train_df['tokens_A'] = train_df['clean_A'].apply(lambda x: x.split())
train_df['tokens_B'] = train_df['clean_B'].apply(lambda x: x.split())
train_df['tokens_C'] = train_df['clean_C'].apply(lambda x: x.split())
train_df['tokens_D'] = train_df['clean_D'].apply(lambda x: x.split())
train_df['tokens_E'] = train_df['clean_E'].apply(lambda x: x.split())

all_tokens = pd.concat([
    train_df['tokens_prompt'], train_df['tokens_A'], 
    train_df['tokens_B'], train_df['tokens_C'], 
    train_df['tokens_D'], train_df['tokens_E']
]).tolist()

w2v_model = Word2Vec(sentences=all_tokens, vector_size=100, window=5, min_count=1, workers=4)

def get_sentence_embedding(tokens, model, vector_size):
    valid_words = [word for word in tokens if word in model.wv]
    if not valid_words:
        return np.zeros(vector_size)
    

    return np.mean([model.wv[word] for word in valid_words], axis=0)

def get_top_3_w2v(row, model):
    vector_size = model.vector_size
    prompt_vec = get_sentence_embedding(row['tokens_prompt'], model, vector_size).reshape(1, -1)
    
    options_tokens = [row['tokens_A'], row['tokens_B'], row['tokens_C'], row['tokens_D'], row['tokens_E']]
    labels = ['A', 'B', 'C', 'D', 'E']
    
    similarities = []
    for opt_tokens in options_tokens:
        opt_vec = get_sentence_embedding(opt_tokens, model, vector_size).reshape(1, -1)
        # Handle cases where vectors are all zeros
        if not np.any(prompt_vec) or not np.any(opt_vec):
            sim = 0.0
        else:
            sim = cosine_similarity(prompt_vec, opt_vec)[0][0]
        similarities.append(sim)
        
    top_3_idx = np.argsort(similarities)[::-1][:3]
    top_3_labels = [labels[i] for i in top_3_idx]
    
    return " ".join(top_3_labels)

train_df['w2v_prediction'] = train_df.apply(lambda row: get_top_3_w2v(row, w2v_model), axis=1)

w2v_map3 = calculate_map_at_3(train_df['answer'], train_df['w2v_prediction'])
print(f"Baseline Word2Vec MAP@3 Score: {w2v_map3:.4f}")

Baseline Word2Vec MAP@3 Score: 0.3233
